# 4-City TSP with PennyLane

Encodes TSP as a QUBO with one-hot constraints, then solves it variationally with a hardware-efficient ansatz. City 0 is pinned at slot 0, leaving 3x3 = 9 binary variables.

In [ ]:
import itertools
import numpy as np
import pennylane as qml
import scipy.optimize as opt

## Problem definition

In [ ]:
N_CITIES = 4
N_SLOTS = N_CITIES - 1
N_VARS = N_SLOTS * N_SLOTS  # 9
NAMES = ("depot", "harbor", "market", "tower")
DIST = np.array([
    [0.0, 2.0, 3.0, 2.5],
    [2.0, 0.0, 1.5, 4.0],
    [3.0, 1.5, 0.0, 1.0],
    [2.5, 4.0, 1.0, 0.0],
])
PENALTY = 10.0

def var_index(city, slot):
    return (city - 1) * N_SLOTS + (slot - 1)

def decode_bits(bits):
    slots = [0, -1, -1, -1]
    used = set()
    for city in range(1, N_CITIES):
        ones = [s for s in range(1, N_CITIES) if bits[var_index(city, s)] == 1]
        if len(ones) != 1:
            return None
        s = ones[0]
        if s in used:
            return None
        slots[s] = city
        used.add(s)
    return slots

def tour_cost(tour):
    return sum(DIST[tour[k], tour[(k+1) % len(tour)]] for k in range(len(tour)))

def bitstring_cost(bits):
    tour = decode_bits(bits)
    if tour is None:
        return PENALTY * N_CITIES
    return tour_cost(tour)

## Classical baseline

In [ ]:
best_cost = float("inf")
for perm in itertools.permutations(range(1, N_CITIES)):
    tour = (0,) + perm
    c = tour_cost(list(tour))
    if c < best_cost:
        best_cost = c
        best_tour = tour
    print(f"  {' -> '.join(NAMES[i] for i in tour)}  cost={c:.1f}")
print(f"\nOptimal: {' -> '.join(NAMES[i] for i in best_tour)}  cost={best_cost:.1f}")

## Quantum ansatz

In [ ]:
dev = qml.device("default.qubit", wires=N_VARS)

@qml.qnode(dev, diff_method="parameter-shift")
def tsp_circuit(params):
    n_params = len(params)
    n_layers = n_params // (2 * N_VARS)
    p = 0
    for i in range(N_VARS):
        qml.RY(np.pi * 0.5, wires=i)
    for _ in range(n_layers):
        for i in range(N_VARS):
            qml.RX(params[p], wires=i)
            p += 1
        for i in range(N_VARS):
            qml.RZ(params[p], wires=i)
            p += 1
        for i in range(N_VARS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_VARS - 1, 0])
    return qml.probs(wires=range(N_VARS))

def cost_function(params):
    probs = tsp_circuit(params)
    expected = 0.0
    for idx, p_val in enumerate(probs):
        if p_val < 1e-12:
            continue
        bits = [(idx >> q) & 1 for q in range(N_VARS)]
        expected += p_val * bitstring_cost(bits)
    return expected

## Optimise

In [ ]:
n_layers = 3
n_params = n_layers * 2 * N_VARS
rng = np.random.default_rng(7)
init = rng.uniform(-np.pi, np.pi, size=n_params)

result = opt.minimize(cost_function, init, method="COBYLA",
                      options={"maxiter": 150, "rhobeg": 0.3})
print(f"Converged: {result.success}")
print(f"Expected cost: {result.fun:.4f}")

## Results

In [ ]:
probs = tsp_circuit(result.x)
top_indices = np.argsort(probs)[-5:][::-1]
print("Top 5 outcomes:")
for idx in top_indices:
    p_val = probs[idx]
    bits = [(idx >> q) & 1 for q in range(N_VARS)]
    tour = decode_bits(bits)
    if tour is not None:
        c = tour_cost(tour)
        route = " -> ".join(NAMES[i] for i in tour)
        print(f"  |{format(idx, f'0{N_VARS}b')}⟩  P={p_val:.4f}  route={route}  cost={c:.1f}")
    else:
        print(f"  |{format(idx, f'0{N_VARS}b')}⟩  P={p_val:.4f}  (invalid)")